<a href="https://www.udea.edu.co">
    <img src="https://www.udea.edu.co/wps/wcm/connect/udea/2288a382-341c-41ee-9633-702a83d5ad2b/logosimbolo-horizontal-png.png?MOD=AJPERES&CVID=ljeSAX9" alt="Universidad de Antioquia Logo" width="200"/>
</a>


**SISTEMAS DE CONTROL CONTÍNUO**

- Prof. Hernán Felipe García Arias, PhD
- Prof. Andrés Felipe López García, PhD
- Facultad de Ingeniería
- Ingeniería Electrónica
- UdeA - 2026

<h2>TCLab Historian</h2>

<h3>Registro básico</h3>
<p>La clase <code>tclab.Historian</code> proporciona funcionalidad de registro de datos. Dado un objeto <code>TCLab</code>, se crea un historizador con:</p>
<pre><code>import tclab
lab = tclab.TCLab()
h = tclab.Historian(lab.sources)</code></pre>
<p>El historizador inicializa un registro de datos. Las fuentes del registro se especifican en el argumento de <code>tclab.Historian</code>. Un conjunto de fuentes por defecto para la instancia <code>lab</code> se encuentra en <code>lab.sources</code>.</p>
<p>Para actualizar el registro de datos, utilice:</p>
<pre><code>h.update(t)</code></pre>
<p>Donde <code>t</code> es el tiempo actual del reloj. Si se omite, el historizador calculará automáticamente su propio tiempo.</p>

In [ ]:
import tclab

TCLab = tclab.setup(connected=False, speedup=10)

with TCLab() as lab:
    h = tclab.Historian(lab.sources)
    for t in tclab.clock(20):
        lab.Q1(100 if t <= 10 else 0)
        print("Time:", t, 'seconds')
        h.update(t)

<h2>Acceso al registro de datos desde Historian</h2>
<p><code>Historian</code> mantiene un registro de datos que se actualiza cada vez que se llama a <code>h.update()</code>. La lista de variables registradas por un Historian se obtiene de su atributo <code>sources</code>, por ejemplo:</p>
<pre><code>variables = h.sources  # Lista de (nombre, función) a registrar</code></pre>
<p>Cada elemento de <code>variables</code> es una tupla (<em>nombre</em>, <em>función de muestreo</em>) que indica qué datos se incluyen en el log.</p>

In [ ]:
h.columns

%%html
<h2>Acceso a series temporales desde Historian</h2>
<p>Las series temporales individuales están disponibles en <code>h.fields</code>. Para el conjunto predeterminado de fuentes, puede obtenerse así:</p>
<pre><code>t, T1, T2, Q1, Q2 = h.fields</code></pre>
<p>Por ejemplo, para graficar la evolución de la temperatura <code>T1</code> frente al tiempo:</p>
<pre><code>import matplotlib.pyplot as plt

t, T1, _, _, _ = h.fields  # Desempaquetar solo t y T1

plt.plot(t, T1)
plt.xlabel('Tiempo (s)')
plt.ylabel('Temperatura T1 (°C)')
plt.title('Histórico de T1')
plt.grid(True)
plt.show()</code></pre>


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

t, T1, T2, Q1, Q2 = h.fields
plt.plot(t, T1)
plt.xlabel('Time / seconds')
plt.ylabel('Temperature / °C')
plt.grid()

### Accessing fields by name

To enable easy access of a particular field by name, `Historian` allows individual field access via the `logdict` dictionary. It also has a `t` property for quick access of the time field, so the above code could also have been written this way:

In [ ]:
%matplotlib notebook
import matplotlib.pyplot as plt

column = 'T1'
plt.plot(h.t, h.logdict['T1'])
plt.xlabel('Time / seconds')
plt.ylabel('Temperature / °C')
plt.grid()

Here is a simple code using these access methods to plot all the fields an `Historian` is logging:

In [ ]:
def plotlog(historian):
    t = historian.t
    fig, axes = plt.subplots(nrows=len(historian.fields) - 1,
                        ncols=1, sharex='col')
    for axis, column in zip(axes, historian.columns[1:]):
        axis.step(t, historian.logdict[column], where='post')
        axis.grid()
        axis.set_ylabel(column)
    plt.xlabel('Time / Seconds')

plotlog(h)

### Tabular access

The entire data history is available from the historian as the attribute `.log`. Here we show the first three rows from the log. The columns are returned in the same order as `.columns`.

In [ ]:
h.columns

In [ ]:
h.log[:3]

## Saving to a file
The log can be saved to a CSV file using `.to_csv()`. This is useful for spreadsheet access.

In [ ]:
h.to_csv('saved_data.csv')

## Accessing log data using Pandas

Pandas is a widely use Python library for manipulation and analysis of data sets. Here we show how to access the `tclab.Historian` log using Pandas.

The log can be converted to a Pandas dataframe.

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(h.log, columns=h.columns, index='Time')
df.head()

The following cells provide examples of plots that can be constructed once the data log has been converted to a pandas dataframe.

In [ ]:
df.plot()

In [ ]:
df[['T1','T2']].plot(grid=True)

## Specifying Sources for `tclab.Historian`

To create an instance of `tclab.Historian`, a set of sources needs to be specified. For many cases the default sources created for an instance of `TCLab` is sufficient. However, it is possible to specify additional sources which can be useful when implementing more complex algorithms for process control.

`sources` is specified as a list of tuples. Each tuple as two elements. The first element is a label for the source. The second element is a function that returns a value.

The following cell shows how to create a source with the label `Power` with a value equal to the estimated heater power measured in watts. This is created on the assumption that 100% of a maximum power of 200 corresponds to 4.2 watts.

In [ ]:
import tclab

TCLab = tclab.setup(connected=False, speedup=10)

with TCLab() as lab:
    sources = [
        ('T1', lambda: lab.T1),
        ('Power', lambda: lab.P1*lab.U1*4.2/(200*100))
    ]
    h = tclab.Historian(sources)
    for t in tclab.clock(20):
        lab.Q1(100 if t <= 10 else 0)
        print("Time:", t, 'seconds')
        h.update(t)

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(h.log, columns=h.columns, index='Time')
df.head()

### Functions with multiple returns

In some cases it is easier to calculate a number of different variables to be logged in one function, especially if intermediate results are used in following calculations. This can be accommodated by `Historian` by passing `None` as the function for subsequent values if a previous value returned a list of values.

In [ ]:
import tclab

TCLab = tclab.setup(connected=False, speedup=10)

def log_values():
    T1 = lab.T1
    T1Kelvin = T1 + 273.15
    power = lab.P1*lab.U1*4.2/(200*100)
    return T1, T1Kelvin, power

with TCLab() as lab:
    h = tclab.Historian([('T1', log_values),
                         ('T1Kelvin', None),
                         ('Power', None)])
    for t in tclab.clock(20):
        lab.Q1(100 if t <= 10 else 0)
        print("Time:", t, 'seconds')
        h.update(t)

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(h.log, columns=h.columns, index='Time')
df.head()

In [ ]:
h.log

## Sessions

It is possible to run multiple experiments using the same `Historian` and page back to them after running them:

In [ ]:
import tclab

TCLab = tclab.setup(connected=False, speedup=10)

with TCLab() as lab:
    h = tclab.Historian(lab.sources)
    for t in tclab.clock(20):
        lab.Q1(100 if t <= 10 else 0)
        h.update(t)
    h.new_session()
    for t in tclab.clock(20):
        lab.Q1(0 if t <= 10 else 100)
        h.update(t)

To see the stored sessions, use `get_sessions`:

In [ ]:
h.get_sessions()

The historian log shows the data for the last session, where the heater started open

In [ ]:
h.log[:2]

To roll back to a different session, use `load_session()`:

In [ ]:
h.load_session(1)

In [ ]:
h.log[:2]

## Persistence

`Historian` stores results in a [SQLite](https://www.sqlite.org/) database. By default, it uses the special file `:memory:` which means the results are lost when you shut down or restart the kernel. To retain values across runs, you can specify a filename (by convention SQLite databases have the `.db` extension).

In [ ]:
import tclab

TCLab = tclab.setup(connected=False, speedup=10)

with TCLab() as lab:
    h = tclab.Historian(lab.sources, dbfile='test.db')
    for t in tclab.clock(20):
        lab.Q1(100 if t <= 10 else 0)
        h.update(t)

Every time you run this cell, new sessions will be added to the file. These sessions can be loaded as shown in the Sessions section above. There is currently no support for managing sessions. If you want to remove the old sessions, delete the database file.